# Calendar Dimension Table

Builds the standard calendar dimension `adb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including date keys and year, quarter, month, week and day attributes.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  -- Keys
  CAST(date_format(date, 'yyyyMMdd') AS INT)                         AS date_key,
  date                                                               AS date,

  -- Year
  year(date)                                                         AS year,
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END                       AS half_year,
  CONCAT('H', CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END)          AS half_year_name,
  ((year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0) AS is_leap_year,
  make_date(year(date), 1, 1)                                        AS year_start_date,
  make_date(year(date), 12, 31)                                      AS year_end_date,

  -- Quarter
  quarter(date)                                                      AS quarter,
  CONCAT('Q', quarter(date))                                         AS quarter_name,
  CONCAT(year(date), '-Q', quarter(date))                            AS year_quarter,
  CAST(date_trunc('QUARTER', date) AS DATE)                          AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS quarter_end_date,
  datediff(date, CAST(date_trunc('QUARTER', date) AS DATE)) + 1      AS day_of_quarter,

  -- Month
  month(date)                                                        AS month,
  date_format(date, 'MMMM')                                          AS month_name,
  date_format(date, 'MMM')                                           AS month_short_name,
  date_format(date, 'yyyy-MM')                                       AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                           AS year_month_key,
  CAST(date_trunc('MONTH', date) AS DATE)                            AS month_start_date,
  last_day(date)                                                     AS month_end_date,
  day(last_day(date))                                                AS days_in_month,

  -- Week (ISO 8601: weeks start Monday)
  weekofyear(date)                                                   AS week_of_year,
  extract(YEAROFWEEK FROM date)                                      AS iso_year,
  CONCAT(extract(YEAROFWEEK FROM date), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  CAST(ceil(day(date) / 7.0) AS INT)                                 AS week_of_month,
  CAST(date_trunc('WEEK', date) AS DATE)                             AS week_start_date,
  date_add(CAST(date_trunc('WEEK', date) AS DATE), 6)                AS week_end_date,

  -- Day
  day(date)                                                          AS day_of_month,
  dayofyear(date)                                                    AS day_of_year,
  extract(DAYOFWEEK_ISO FROM date)                                   AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(date, 'EEEE')                                          AS day_name,
  date_format(date, 'EEE')                                           AS day_short_name,

  -- Flags
  extract(DAYOFWEEK_ISO FROM date) IN (6, 7)                         AS is_weekend,
  extract(DAYOFWEEK_ISO FROM date) BETWEEN 1 AND 5                   AS is_weekday,
  date = CAST(date_trunc('MONTH', date) AS DATE)                     AS is_month_start,
  date = last_day(date)                                              AS is_month_end,
  date = CAST(date_trunc('QUARTER', date) AS DATE)                   AS is_quarter_start,
  date = date_sub(add_months(CAST(date_trunc('QUARTER', date) AS DATE), 3), 1) AS is_quarter_end,
  (month(date) = 1 AND day(date) = 1)                                AS is_year_start,
  (month(date) = 12 AND day(date) = 31)                              AS is_year_end
FROM dates
ORDER BY date

In [0]:
%sql
-- Expected: 33,238 rows (91 years x 365 + 23 leap days), unique date_key, range 1940-01-01 to 2030-12-31
SELECT
  COUNT(*)                 AS total_rows,
  COUNT(DISTINCT date_key) AS distinct_date_keys,
  MIN(date)                AS min_date,
  MAX(date)                AS max_date,
  SUM(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM adb_caresync.silver.calendar

In [0]:
%sql
SELECT *
FROM adb_caresync.silver.calendar
WHERE date BETWEEN DATE'2026-12-28' AND DATE'2027-01-04'
ORDER BY date